In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [6]:
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

# Xử lý các giá trị NaN
train_df['sarcasm_type'] = train_df['sarcasm_type'].fillna("None")
test_df['sarcasm_type'] = test_df['sarcasm_type'].fillna("None")

# CHIẾN LƯỢC CÂN BẰNG DỮ LIỆU (RESAMPLING)
print("Tỉ lệ trước khi cân bằng:\n", train_df['sarcasm_label'].value_counts())

# Tách nhóm
df_non = train_df[train_df['sarcasm_label'] == 'Non-Sarcastic']
df_sar = train_df[train_df['sarcasm_label'] == 'Sarcastic']

# 1. Undersample: Giảm Non-Sarcastic xuống 1500 mẫu
df_non_sampled = df_non.sample(n=1500, random_state=42)

# 2. Oversample: Nhân bản từng Sarcasm Type lên tối thiểu 150 mẫu
oversampled_list = []
for s_type in df_sar['sarcasm_type'].unique():
    df_type = df_sar[df_sar['sarcasm_type'] == s_type]
    target_count = max(150, len(df_type)) # Nếu đã > 150 thì giữ nguyên, nhỏ hơn thì nhân bản
    df_type_sampled = df_type.sample(n=target_count, replace=True, random_state=42)
    oversampled_list.append(df_type_sampled)

df_sar_balanced = pd.concat(oversampled_list)

# 3. Gộp lại và xáo trộn ngẫu nhiên
train_df = pd.concat([df_non_sampled, df_sar_balanced]).sample(frac=1.0, random_state=42).reset_index(drop=True)
print("Tỉ lệ sau khi cân bằng:\n", train_df['sarcasm_type'].value_counts())

# TẠO NHÃN 6 CLASS CHO BÀI TOÁN PHÂN LOẠI
def get_6_labels(row):
    if row['sarcasm_label'] == 'Non-Sarcastic': return 'Non-Sarcastic'
    if pd.isna(row['sarcasm_type']) or row['sarcasm_type'] == 'None': return 'Propositional Contradiction'
    return row['sarcasm_type']

train_df['final_label'] = train_df.apply(get_6_labels, axis=1)
test_df['final_label'] = test_df.apply(get_6_labels, axis=1)

le = LabelEncoder()
y_train = le.fit_transform(train_df['final_label'])
y_test = le.transform(test_df['final_label'])

# Nối Text
X_train_text = "Ngữ cảnh: " + train_df['video_core_content'].astype(str) + " Bình luận: " + train_df['comment'].astype(str)
X_test_text = "Ngữ cảnh: " + test_df['video_core_content'].astype(str) + " Bình luận: " + test_df['comment'].astype(str)


Tỉ lệ trước khi cân bằng:
 sarcasm_label
Non-Sarcastic    6639
Sarcastic         523
Name: count, dtype: int64
Tỉ lệ sau khi cân bằng:
 sarcasm_type
None                           1500
Propositional Contradiction     180
Lexical Contradiction           171
Hypothetical                    150
Rhetorical Question             150
Wh-Question                     150
Name: count, dtype: int64


In [7]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [8]:
svm_model = SVC(kernel='linear', class_weight='balanced', random_state=42)
svm_model.fit(X_train, y_train)

# 4. DỰ ĐOÁN VÀ ĐÁNH GIÁ
y_pred = svm_model.predict(X_test)
target_names = le.inverse_transform([0, 1, 2, 3, 4, 5])

print("="*60)
print("🏆 BÁO CÁO KẾT QUẢ SVM + TF-IDF")
print("="*60)
print(classification_report(y_test, y_pred, target_names=target_names))

🏆 BÁO CÁO KẾT QUẢ SVM + TF-IDF
                             precision    recall  f1-score   support

               Hypothetical       0.10      0.42      0.16        12
      Lexical Contradiction       0.06      0.37      0.10        19
              Non-Sarcastic       0.99      0.62      0.76       738
Propositional Contradiction       0.09      0.45      0.15        20
        Rhetorical Question       0.09      1.00      0.16         5
                Wh-Question       0.25      0.50      0.33         2

                   accuracy                           0.60       796
                  macro avg       0.26      0.56      0.28       796
               weighted avg       0.93      0.60      0.71       796



In [9]:
predicted_merged = le.inverse_transform(y_pred)

# Tách ngược lại thành Label và Type để đồng nhất format với LLM
test_df['predicted_label'] = ['Non-Sarcastic' if p == 'Non-Sarcastic' else 'Sarcastic' for p in predicted_merged]
test_df['predicted_type'] = ['None' if p == 'Non-Sarcastic' else p for p in predicted_merged]

output_file = "test_result_SVM.csv"
test_df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu kết quả chi tiết của SVM vào: {output_file}")

✅ Đã lưu kết quả chi tiết của SVM vào: test_result_SVM.csv
